<a href="https://colab.research.google.com/github/Emelygabrielaa/ICVC/blob/main/1_Calgary_Building_Permits%2C_Descriptive_Analysis_and_EDA_Phase_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project: Calgary Building Permits, Descriptive Analysis and EDA**

**Path:** Statistician path, Project 1 of 11  
**Author:** Emely Rodriguez  
**Languages:** Python (lead), R (cross-check in a separate notebook)  
**Data as of:** 2026-10-27 (502,814 rows). The portal showed the data as last updated on 2026-10-27.  
**Licence:** Contains information licensed under the Open Government Licence - City of Calgary.



## Project phases

| Phase | Content | Status |
|---|---|---|
| **0. Data audit** | **Structure, quality and definitions of the source data** | **This notebook** |
| 1. Analysis dataset | Handling of audit issues, selected columns, derived variables, dated snapshot | Pending |
| 2. Part A | Descriptive answers to the five questions | Pending |
| 3. Part B | Formal tests and models | Pending |
| 4. Report | Findings and conclusions | Pending |

## What this project is about
The City of Calgary publishes its building permit application records, which start in June 1999. In this project I use that table to answer five practical questions: how long applications take, how much the projects are worth, how activity changes over time, where it happens, and how complete the data is. Part A answers them with descriptive statistics. Part B goes back to the same questions with formal tests and models.

## The five questions

| # |Topic | Question | Part A: descriptive | Part B: going deeper |
|---|---|---|---|---|
| 1 | Processing time | How long do applications take? | Days from application to issue, with percentiles by permit type | Kruskal-Wallis with pairwise tests, and Kaplan-Meier curves |
| 2 | Project cost | How much are the projects worth? | Skewness, log scale, tail, and summaries by class | Lognormal fit check, bootstrap intervals for medians, quantile regression |
| 3 | Volume over time | How does activity change over time? | Counts by year and month, seasonality plots | Decomposition, Mann-Kendall trend test, count regression |
| 4 | Geography | Where does it happen? | Counts and medians by community, and a map | Rates per community and Moran's I |
| 5 | Missingness | How complete is the data? | Missing rates by column and permit type | Chi-square tests and logistic regression of missingness |

## The data

- **Source:** City of Calgary Open Data portal, dataset "Building Permits" (ID c2es-76ed)
- **Provider:** The City of Calgary, Dev Business and Bldg Srvc
- **What a row is:** a building permit application record, identified by `permitnum` (unique, checked in section 0.2.1)
- **Coverage:** applications from 1999-06-22 onward, updated daily
- **Size:** about 503,000 rows and 30 columns
- **Coordinates:** WGS84 (EPSG:4326), in degrees
- **Licence:** City of Calgary Open Data Terms (https://data.calgary.ca/d/Open-Data-Terms/u45n-7awa)


**Reproducibility note:** the table is updated daily and Phase 0 queries it directly, so its figures describe the version available on the "Data as of" date. From Phase 1 onward, all results are computed on a dated snapshot saved in the project repository.

## Pills for this project
**Board pills:** Descriptive/EDA, Group Comparison, Missing-data analysis, Trend analysis,
Regression analysis, Uncertainty analysis, Spatial

**Method pills:** percentiles, ecdf, skewness, log-transform, heavy-tails, kruskal-wallis,
multiple-comparisons, censoring, kaplan-meier, bootstrap, quantile-regression, seasonality,
decomposition, mann-kendall, count-regression, rates-per-area, morans-i, choropleth,
missing-rates, chi-square, logistic-regression, mar-vs-mcar

--------

# **Phase 0: Data audit**

**Goal:** understand what each row and column means, check the quality of the data, and define key terms before any analysis.

**Outputs of this phase:**
- A data dictionary for the 30 columns
- A list of data quality findings (the issues log)
- Definitions needed later, such as what counts as "time to issue"

**Approach:** the portal can count and summarize the table on its own server, through its SODA API. This phase uses that to inspect the full table without downloading its roughly 503,000 rows yet.

No data is modified in this phase. Findings are recorded here, and the decisions on how to handle them are made in Phase 1.

## 0.0 Setup

The setup has two parts: the libraries and the portal address, and a helper function that sends queries to the portal.

### 0.0.1 Libraries and portal address

`requests` sends the web request, `pandas` stores the results as tables, and `StringIO` lets pandas read the text the portal returns. `BASE` is the address of the dataset's data endpoint, in CSV format.

In [40]:
# Import the libraries: web requests, data tables, and a text reader for the portal's response
import requests, pandas as pd
from io import StringIO

# Address of the dataset's data endpoint (CSV format)
BASE = "https://data.calgary.ca/resource/c2es-76ed.csv"

### 0.0.2 Helper function

`soda()` sends a query to the portal and returns the answer as a table (a pandas DataFrame). If the portal rejects a request, `raise_for_status()` stops the code with a clear error.

Queries are written in SoQL, the portal's SQL-like language, using these parameters:

- `$select`: the columns or calculations to return
- `$where`: a filter
- `$group`: groups rows before counting
- `$order`: sorts the result
- `$limit`: the maximum number of rows to return (the portal returns 1,000 if none is given)

Parameter names start with `$`, so they are passed to the function as a dictionary, for example `soda(**{"$select": "count(*) as n"})`.

The portal's CSV is read with the pandas defaults, which show text values such as "N/A" or "NA" as missing values (`NaN`). When the difference matters, the raw text is read directly.

In [41]:
# Send a query to the portal and return the answer as a pandas DataFrame
def soda(**params):
    r = requests.get(BASE, params=params, timeout=120)  # send the request, waiting up to 120 seconds
    r.raise_for_status()                                # stop with an error if the portal rejects it
    return pd.read_csv(StringIO(r.text))                # read the CSV text into a table

## 0.1 Snapshot

The portal lists the table as updated daily, but rows were added while this notebook was being built, so this cell records the date of the check and the total number of rows. Because the table changes, the notebook is run in one pass, and every result in Phase 0 refers to the version of the table available at that moment. The date comes from the Colab server clock, which uses UTC.

In [42]:
# Count the rows in the table and print the date of this check
total = soda(**{"$select": "count(*) as n"})
print("As of", pd.Timestamp.today().date(), "| rows:", total["n"][0])

As of 2026-10-27 | rows: 502814


**Result:** as of 2026-10-27 the table has 502,814 rows. The portal lists its last update as 2026-10-27.

## 0.2 Dataset structure

Before looking at the content of any column, the first step is to establish the basic structure of the table: how many records it holds, how many columns it has, and the data type the portal assigns to each one. The row count comes from the query in 0.1. The column names and types come from the metadata that the portal publishes for the dataset.

In [43]:
# Download the dataset metadata published by the portal (column names and types)
meta = requests.get("https://data.calgary.ca/api/views/c2es-76ed.json", timeout=120).json()

# Build a table with one row per column, ignoring hidden system fields (names starting with ":")
cols = pd.DataFrame([{"column": c["fieldName"], "type": c["dataTypeName"]}
                     for c in meta["columns"] if not c["fieldName"].startswith(":")])

# Show the number of rows (from 0.1) and columns, then the list of columns with their types
print("Rows:", total["n"][0], "| Columns:", len(cols))
display(cols)

Rows: 502814 | Columns: 30


,column,type
0,permitnum,text
1,statuscurrent,text
2,applieddate,calendar_date
3,issueddate,calendar_date
4,completeddate,calendar_date
5,permittype,text
6,permittypemapped,text
7,permitclass,text
8,permitclassgroup,text
9,permitclassmapped,text


**Result:**
- The table has 502,814 rows and 30 columns.
- By portal data type: 20 text, 6 number (`housingunits`, `estprojectcost`, `totalsqft`, `latitude`, `longitude`, `locationcount`), 3 calendar date (`applieddate`, `issueddate`, `completeddate`) and 1 point (`point`).

**Notes:**
- The three raw classification columns (`permittype`, `permitclass`, `workclass`) each have a grouped version (`...group`) and a mapped version (`...mapped`). Their relationship is reviewed in section 0.13.
- Location is stored in several forms: `latitude` and `longitude`, `point`, and the text columns `locationswkt` and `locationsgeojson`. Many applications link to more than one location (`locationcount`), as shown in section 0.12.
- `applicantname` and `contractorname` are free text fields. They are not used in the analysis.

### 0.2.1 Uniqueness of the permit number

The analysis assumes that each row represents one permit application, identified by `permitnum`. This assumption is verified by comparing the total number of rows with the number of distinct permit numbers. If the two figures differ, the repeated values would be inspected to understand why.

In [44]:
# Compare the total number of rows with the number of distinct permit numbers
display(soda(**{"$select": "count(*) as n_rows, count(distinct permitnum) as n_distinct"}))

,n_rows,n_distinct
0,502814,502814


In [45]:
# List the permit numbers that appear more than once (top 10 by number of repetitions)
display(soda(**{"$select": "permitnum, count(*) as n", "$group": "permitnum",
                "$having": "count(*) > 1", "$order": "n DESC", "$limit": 10}))

,permitnum,n


**Result:** the table has 502,814 rows and 502,814 distinct permit numbers. No `permitnum` is repeated and none is missing.

**Notes:**
- Each row corresponds to one application, so counts of rows can be read directly as counts of applications.
- This holds for the version of the table available at the date shown in 0.1. The check is repeated on the analysis dataset in Phase 1.

## 0.3 Status of each application

Every application has a current status. Counting the rows per status shows how many applications finished, how many ended without a permit, and how many are still in progress. This count determines how "time to issue" is defined for Question 1.

In [46]:
# Count the applications in each current status, from most to least frequent
display(soda(**{"$select": "statuscurrent, count(*) as n",
                "$group": "statuscurrent", "$order": "n DESC"}))

,statuscurrent,n
0,Completed,455111
1,Cancelled,14886
2,Issued Permit,11036
3,Expired,7611
4,File Closed,4463
5,Pre Backfill Phase,2404
6,Hold,1903
7,Pending Plans Review Assignment,1035
8,In Review,914
9,Refused,874


**Result:**
- There are 24 distinct statuses. The five largest account for 98.1% of the table: Completed (455,111; 90.5%), Cancelled (14,886; 3.0%), Issued Permit (11,036; 2.2%), Expired (7,611; 1.5%) and File Closed (4,463; 0.9%).
- The other 19 statuses are each below 0.5%. Several describe applications still in progress, such as `New`, `In Review` and the `Pending` and `Hold` statuses.

**Notes:**
- The portal does not define the statuses. Their role is examined in section 0.14.
- Some applications ended without a permit (`Cancelled`, `Refused`) and others are still in progress. A time to issue can only be measured for applications that were issued, which is taken into account in Question 1.

## 0.4 Date coverage

Each application has up to three dates: applied, issued, and completed. The earliest and latest value of each shows the time window the table covers. The latest dates also mark where the observation window ends, which matters for applications that are still open (Question 1) and for incomplete periods in the volume analysis (Question 3).

In [47]:
# Earliest and latest value of each of the three date columns
display(soda(**{"$select": "min(applieddate) as applied_min, max(applieddate) as applied_max, "
                           "min(issueddate) as issued_min, max(issueddate) as issued_max, "
                           "min(completeddate) as completed_min, max(completeddate) as completed_max"}))

,applied_min,applied_max,issued_min,issued_max,completed_min,completed_max
0,1999-06-22T00:00:00.000,2026-10-26T00:00:00.000,1999-06-23T00:00:00.000,2026-10-26T00:00:00.000,1999-06-30T00:00:00.000,2026-10-26T00:00:00.000


**Result:**

- Applied dates run from 1999-06-22 to 2026-10-26.
- Issued dates run from 1999-06-23 to 2026-10-26.
- Completed dates run from 1999-06-30 to 2026-10-26.
- Every date is stored at midnight, with no time of day, so the time between two dates is measured in whole days.

**Implications:**

- The data is current to the day before the check, so the most recent weeks are incomplete.
- Applications still open are only observed up to the extraction date (right-censored). This matters for Question 1.
- 1999 is a partial year because the table starts in June, and 2026 is partial because it ends in October. Both matter for Question 3.

### 0.4.1 Order of the dates

Time to issue is the number of days between the application date and the issue date, so it assumes that the dates come in a logical order: applied, then issued, then completed. This check counts the applications where the order is reversed, which would produce negative times. It also counts applications issued on the same day they were submitted, which give a time of zero days. That is possible and is not an error, but it affects how the distribution is read.

In [48]:
# Count the applications whose dates are out of order, and those issued on the day of application
checks = [("issued before applied", "issueddate < applieddate"),
          ("completed before issued", "completeddate < issueddate"),
          ("completed before applied", "completeddate < applieddate"),
          ("issued on the same day as applied", "issueddate = applieddate")]
for label, where in checks:
    print(label, soda(**{"$select": "count(*) as n", "$where": where})["n"][0])

issued before applied 1
completed before issued 31
completed before applied 1
issued on the same day as applied 105304


**Result:**
- 1 application has an issue date earlier than its application date, 31 have a completion date earlier than the issue date, and 1 has a completion date earlier than the application date.
- 105,304 applications (20.9% of the table) were issued on the same day they were submitted, which gives a time to issue of zero days.

**Implications:**
- The few applications with dates out of order would produce negative times and need a decision before the analysis.
- The large share of same day issues puts a spike at zero in the distribution of time to issue. This affects percentiles, and any log transformation needs to handle zeros. This matters for Question 1.

## 0.5 Missing values in key columns

Counting the missing values in the key columns gives a first view of how complete the data is (Question 5). A missing value is not always an error: it can mean "not applicable", "has not happened yet", or "not recorded". Later sections separate these cases.

In [49]:
# Count the missing values in each key column
for col in ["applieddate", "issueddate", "completeddate", "estprojectcost", "totalsqft",
            "housingunits", "communityname", "latitude"]:
    n = soda(**{"$select": "count(*) as n", "$where": f"{col} IS NULL"})["n"][0]
    print(f"{col}: {n} missing")

applieddate: 0 missing
issueddate: 18863 missing
completeddate: 19852 missing
estprojectcost: 39023 missing
totalsqft: 346307 missing
housingunits: 0 missing
communityname: 0 missing
latitude: 0 missing


**Result:**

| Column | Missing rows | Share of rows |
|---|---|---|
| `applieddate` | 0 | 0% |
| `issueddate` | 18,863 | 3.8% |
| `completeddate` | 19,852 | 3.9% |
| `estprojectcost` | 39,023 | 7.8% |
| `totalsqft` | 346,307 | 68.9% |
| `housingunits` | 0 | 0% |
| `communityname` | 0 | 0% |
| `latitude` | 0 | 0% |

**Notes:**
- `totalsqft` has the highest share of missing values. The data show that it is recorded only for Single Construction Permit records, so most of this missingness is expected. A later section examines this.
- `applieddate`, `housingunits`, `communityname` and `latitude` have no missing values, but complete does not mean valid. Their values are checked separately, for example the placeholder zeros in `latitude` and the label "N/A" in `communityname` (see 0.11.1).
- Whether the missing dates mean "not yet issued" or "ended without being issued" depends on the status. A later section examines this.

## 0.6 Categories in the main classification columns

Permit type, permit class group, and work class group are the main candidates for grouping applications in the analysis. Counting the categories in each column shows how many groups exist and how the rows are distributed, which determines the level of detail that is practical for comparisons. The code prints the number of categories and the 15 most frequent ones.

In [50]:
# Classification columns: number of categories and the 15 most frequent
for col in ["permittype", "permitclassgroup", "workclassgroup"]:
    d = soda(**{"$select": f"{col}, count(*) as n", "$group": col,
                "$order": "n DESC", "$limit": 5000})
    print(f"\n{col}: {len(d)} categories")
    display(d.head(15))


permittype: 5 categories


,permittype,n
0,Residential Improvement Project,226823
1,Single Construction Permit,156507
2,Commercial / Multi Family Project,101057
3,Demolition,17680
4,Environmental Restoration Permit,747



permitclassgroup: 16 categories


,permitclassgroup,n
0,Single Family,273258
1,Garage,64418
2,Commercial,62472
3,Two Family,28406
4,Secondary Suites,18523
5,Unspecified,13686
6,Townhouse,10889
7,Institutional,9695
8,Apartment,7002
9,Industrial,5596



workclassgroup: 4 categories


,workclassgroup,n
0,New,245941
1,Improvement,234438
2,Demolition,17680
3,Unspecified,4755


**Result:**

- `permittype` (5 categories): Residential Improvement Project (226,823 rows, 45.1%), Single Construction Permit (156,507, 31.1%), Commercial / Multi Family Project (101,057, 20.1%), Demolition (17,680, 3.5%) and Environmental Restoration Permit (747, 0.1%).
- `permitclassgroup` (16 categories): Single Family (273,258, 54.3%), Garage (64,418, 12.8%) and Commercial (62,472, 12.4%) cover about 80% of the rows. The 15 largest categories are shown, and the 16th has 98 rows.
- `workclassgroup` (4 categories): New (245,941, 48.9%), Improvement (234,438, 46.6%), Demolition (17,680, 3.5%) and Unspecified (4,755, 0.9%).

**Notes:**

- The Demolition count is the same in `permittype` and `workclassgroup` (17,680), which is consistent.
- "Unspecified" appears as a category in both class groupings (2.7% of rows in `permitclassgroup`, 0.9% in `workclassgroup`).
- Environmental Restoration Permit has only 747 rows, so comparisons that include it need care.
- One class group is named "Special Function Ten" (3,521 rows), and the portal does not define it.

**Related:** the raw `permitclass` column and the three mapped versions are examined in section 0.13.

## 0.7 Missing dates by status

A missing issued date can mean different things depending on the status: the application may still be waiting for approval, or it may have ended without a permit. Counting the rows with a missing issued date or completed date, by status, separates these cases. This is needed to define "time to issue" and "time to close" for Question 1.

In [51]:
# Missing issued / completed date, by status
for datecol in ["issueddate", "completeddate"]:
    d = soda(**{"$select": "statuscurrent, count(*) as n_missing",
                "$where": f"{datecol} IS NULL",
                "$group": "statuscurrent", "$order": "n_missing DESC"})
    print(f"\nRows with missing {datecol}, by status")
    display(d)


Rows with missing issueddate, by status


,statuscurrent,n_missing
0,Cancelled,12295
1,Hold,1791
2,Pending Plans Review Assignment,1035
3,Refused,873
4,In Review,846
5,New,645
6,Pending Plans Review,484
7,Application Received,336
8,Completed,194
9,Pending Prescreen Review,128



Rows with missing completeddate, by status


,statuscurrent,n_missing
0,Issued Permit,11017
1,Pre Backfill Phase,2404
2,Hold,1893
3,Pending Plans Review Assignment,1035
4,In Review,893
5,Pre Board Phase,800
6,New,645
7,Pending Plans Review,486
8,Application Received,335
9,Pending Prescreen Review,128


**Result:**

Missing issued date (18,863 rows). The rows fall into three groups:

| Group | Rows | Statuses |
|---|---|---|
| Ended without an issue date | 13,168 | Cancelled (12,295) and Refused (873) |
| Still waiting before issue (right-censored) | 5,403 | Hold, Pending Plans Review Assignment, In Review, New, Pending Plans Review, Application Received, and the other pending or hold stages |
| Final-looking status but no issue date (anomalies) | 292 | Completed (194), Expired (85), File Closed (13) |

- Of the 14,886 Cancelled applications, 12,295 (82.6%) have no issue date, so 2,591 were issued before being cancelled.
- Issued Permit, Pre Backfill Phase, Pre Board Phase and Final Phase have no missing issued dates.

Missing completed date (19,852 rows):

- Almost all of them belong to Issued Permit (11,017 of 11,036) and to the stages Pre Backfill Phase (2,404), Pre Board Phase (800) and Final Phase (35), plus the review and hold stages.
- Since these stages have an issue date but no completed date, they appear to sit between issue and completion. This is an inference from the data, not a definition confirmed by the City.
- Almost all Cancelled, Refused, Expired and File Closed rows do have a completed date (only 15 Cancelled, 1 Expired and 1 File Closed lack it, and no Refused), so `completeddate` also records when an application reached a final status, not only when work was completed.
- 25 Completed applications have no completed date, which is also inconsistent with their status.

**Implications for Question 1:**

- "Time to issue" only exists for applications that were issued. Those still waiting are censored, and those that ended without a permit are a competing outcome. The two are treated differently in Part B.
- "Time to close" can use `completeddate` for any status that reached a final state.
- The 292 anomalies are flagged for review in Phase 1.

## 0.8 Missing values in totalsqft

The portal states that `totalsqft` is only available when the permit type is Single Construction Permit. This check counts the rows with a missing `totalsqft` by permit type to test that statement. If it holds, the missingness is structural: it comes from how the data is recorded and not from lost information.

In [52]:
# Rows with missing totalsqft, by permit type
display(soda(**{"$select": "permittype, count(*) as n",
                "$where": "totalsqft IS NULL",
                "$group": "permittype", "$order": "n DESC"}))

,permittype,n
0,Residential Improvement Project,226823
1,Commercial / Multi Family Project,101057
2,Demolition,17680
3,Environmental Restoration Permit,747


**Result:**

| Permit type | Rows with missing `totalsqft` |
|---|---|
| Residential Improvement Project | 226,823 |
| Commercial / Multi Family Project | 101,057 |
| Demolition | 17,680 |
| Environmental Restoration Permit | 747 |
| Total | 346,307 |

- Single Construction Permit does not appear in the list, so none of its 156,507 rows is missing a value.
- For each of the four types listed, the number of missing rows equals the full count of that type in section 0.6. `totalsqft` is therefore missing for every row of those types and for none of the Single Construction Permit rows.

**Conclusion:** the missingness in `totalsqft` is structural, as the portal documentation states. For the four other types the value is "not applicable", not lost. In Question 5 this column is analyzed only within Single Construction Permit, and it is reported separately from true missing values such as `estprojectcost`.


### 0.8.1 Values of totalsqft in Single Construction Permit

For the permit type where `totalsqft` is recorded, this check shows the smallest and largest values and counts the zeros and negative values.

In [53]:
# Smallest and largest totalsqft, and the number of zero and negative values, within Single Construction Permit
display(soda(**{"$select": "min(totalsqft) as sqft_min, max(totalsqft) as sqft_max, "
                           "sum(case(totalsqft = 0, 1, true, 0)) as n_zero, "
                           "sum(case(totalsqft < 0, 1, true, 0)) as n_negative",
                "$where": "permittype = 'Single Construction Permit'"}))

,sqft_min,sqft_max,n_zero,n_negative
0,-15313,41979.25,652,2


**Result:**
- Within Single Construction Permit, `totalsqft` ranges from -15,313 to 41,979.25.
- 652 rows (0.4%) have a value of zero, and 2 rows have a negative value.

**Notes:**
- A negative area is not plausible.
- A zero may mean "not recorded" and not a real area of zero. This is not confirmed.
- The maximum is large for a single construction project but cannot be verified from this table.

### 0.8.2 Values of housingunits

`housingunits` has no missing values. This check shows its smallest and largest values, counts the zeros and negative values, and lists the most frequent values.

In [54]:
# Smallest and largest housingunits, and the number of zero and negative values
display(soda(**{"$select": "min(housingunits) as units_min, max(housingunits) as units_max, "
                           "sum(case(housingunits = 0, 1, true, 0)) as n_zero, "
                           "sum(case(housingunits < 0, 1, true, 0)) as n_negative"}))

,units_min,units_max,n_zero,n_negative
0,-1,715,312857,2


In [55]:
# Most frequent values of housingunits
display(soda(**{"$select": "housingunits, count(*) as n", "$group": "housingunits",
                "$order": "n DESC", "$limit": 10}))

,housingunits,n
0,0,312857
1,1,180073
2,4,2622
3,6,1492
4,5,1007
5,3,728
6,2,714
7,8,604
8,7,480
9,9,230


**Result:**
- `housingunits` ranges from -1 to 715.
- 312,857 rows (62.2%) have 0 units and 180,073 (35.8%) have 1 unit, so 98.0% of the rows have 0 or 1.
- 2 rows have a negative value.
- Among the other values, the most frequent are 4 (2,622 rows), 6 (1,492) and 5 (1,007).

**Notes:**
- Zero is the most common value, which suggests that it means "no housing units" for projects that do not create housing, such as renovations or non-residential work. This is a reading of the data and is not confirmed.
- A negative number of units is not plausible.

## 0.9 Applications per year

Counting applications by the year of the applied date shows the overall volume and how complete each year looks. This is a first look at Question 3 and a check on the partial years identified in section 0.4.

In [56]:
# Count the applications by the year of the applied date
display(soda(**{"$select": "date_extract_y(applieddate) as year, count(*) as n",
                "$group": "date_extract_y(applieddate)", "$order": "year"}))

,year,n
0,1999,6991
1,2000,13551
2,2001,14963
3,2002,17305
4,2003,16634
5,2004,16975
6,2005,19726
7,2006,21007
8,2007,18839
9,2008,15268


**Result:**
- The table has applications from 1999 to 2026, between 13,551 (2000) and 26,521 (2024) per full year.
- 2024 is the highest year in the table. From 2019 (17,373) to 2024, the yearly count rose by about 53%. The 2025 count is 24,207.
- Earlier, the yearly count rose from 13,551 in 2000 to 21,007 in 2006, then fell to 15,268 in 2008, a drop of about 19% from 2007.
- Another decline appears from 19,628 in 2014 to 15,358 in 2016, about 22% lower.

**Partial years:**
- 1999 starts on June 22. Its daily average (about 36 applications per day, 6,991 over 193 days) is close to that of 2000 (about 37 per day), so there is no obvious gap in the early coverage.
- 2026 runs through October 26, so its count (20,412) cannot be compared directly with full years.

**Note:** these are descriptive observations only. Possible causes of the declines and the rise are not examined here.

## 0.10 Estimated project cost

According to the portal, `estprojectcost` is the estimated construction value associated with the permit. It is an estimate and not a final cost. Zeros, negative values, and extreme maximums are common problems in a column like this. This check counts the zeros and negatives and shows the minimum and maximum, before any distribution is described.

In [57]:
# Count the zero and negative estimated costs, and show the minimum and maximum
for label, where in [("cost = 0", "estprojectcost = 0"), ("cost < 0", "estprojectcost < 0")]:
    print(label, soda(**{"$select": "count(*) as n", "$where": where})["n"][0])
display(soda(**{"$select": "min(estprojectcost) as min_cost, max(estprojectcost) as max_cost"}))

cost = 0 0
cost < 0 8


,min_cost,max_cost
0,-135513000,868255213


In [58]:
# List the rows with a negative estimated cost, from the most negative
neg = soda(**{"$select": "permitnum, permittype, statuscurrent, applieddate, estprojectcost",
              "$where": "estprojectcost < 0", "$order": "estprojectcost"})

# Show the date without the time of day and the cost with thousands separators
neg["applieddate"] = pd.to_datetime(neg["applieddate"]).dt.date
display(neg.style.format({"estprojectcost": "{:,.2f}"}))

,permitnum,permittype,statuscurrent,applieddate,estprojectcost
0,BP2004-09155,Commercial / Multi Family Project,Completed,2004-07-12,"-135,513,000.00"
1,BP2008-05122,Commercial / Multi Family Project,Completed,2008-04-30,"-11,186,000.00"
2,BP2010-12171,Commercial / Multi Family Project,Completed,2010-09-20,"-3,500,000.00"
3,BP2008-15062,Single Construction Permit,Completed,2008-12-22,"-1,601,839.35"
4,BP2008-03205,Residential Improvement Project,Completed,2008-03-25,"-16,246.75"
5,BP2011-10352,Residential Improvement Project,Completed,2011-08-18,"-15,797.72"
6,BP2011-15870,Residential Improvement Project,Completed,2011-12-19,-419.99
7,BP2012-04803,Residential Improvement Project,Completed,2012-04-25,-160.00


In [59]:
# List the ten highest estimated costs with their permit details
top = soda(**{"$select": "permitnum, permittype, permitclassgroup, statuscurrent, applieddate, estprojectcost",
              "$where": "estprojectcost IS NOT NULL",
              "$order": "estprojectcost DESC", "$limit": 10})

# Show the date without the time of day and the cost with thousands separators
top["applieddate"] = pd.to_datetime(top["applieddate"]).dt.date
display(top.style.format({"estprojectcost": "{:,.2f}"}))

,permitnum,permittype,permitclassgroup,statuscurrent,applieddate,estprojectcost
0,BP2017-12385,Commercial / Multi Family Project,Institutional,Completed,2017-09-15,"868,255,213.00"
1,BP2007-07297,Commercial / Multi Family Project,Commercial,Completed,2007-05-18,"644,689,500.00"
2,BP2024-24370,Commercial / Multi Family Project,Commercial,Pending Prescreen Review,2024-11-20,"639,048,958.22"
3,BP2011-00001,Commercial / Multi Family Project,Industrial,Completed,2011-01-03,"596,614,379.00"
4,BP2021-09865,Commercial / Multi Family Project,Commercial,Completed,2021-06-04,"400,000,000.00"
5,BP2013-10778,Commercial / Multi Family Project,Commercial,Completed,2013-07-25,"376,000,000.00"
6,BP2009-04369,Commercial / Multi Family Project,Institutional,Completed,2009-05-12,"371,926,500.00"
7,BP2007-15415,Commercial / Multi Family Project,Commercial,Completed,2007-10-04,"320,000,000.00"
8,BP2021-01759,Commercial / Multi Family Project,Commercial,Completed,2021-02-03,"316,961,700.00"
9,BP2026-06022,Commercial / Multi Family Project,Commercial,Pending Plans Review Assignment,2026-04-15,"315,000,000.00"


**Result:**
- No row has an estimated cost of exactly 0.
- 8 rows have a negative estimated cost, from -135,513,000 to -160. All of them are Completed applications from 2004 to 2012: 4 Residential Improvement Project, 3 Commercial / Multi Family Project and 1 Single Construction Permit.
- The maximum is 868,255,213. The ten highest values (315,000,000 and above) all belong to Commercial / Multi Family projects: 7 Commercial, 2 Institutional and 1 Industrial. Eight are Completed, and two belong to applications that are not completed (Pending Prescreen Review and Pending Plans Review Assignment).
- The portal does not state the currency.

**Notes:**
- Negative construction values are not plausible.
- The highest values are very large and cannot be verified from this table. They would have a strong influence on a mean.
- The absence of zeros, together with the 39,023 missing values (7.8%), may mean that projects without an estimate are stored as missing. This is a hypothesis and is not confirmed.

## 0.11 Coordinates

`latitude` and `longitude` have no missing values, but placeholder or wrong values, such as zeros or points outside the city, can still be present. The minimum and maximum of each are compared with the approximate extent of Calgary, which is roughly 50.8 to 51.2 degrees north and 113.9 to 114.3 degrees west.

In [60]:
# Minimum and maximum of latitude and longitude
display(soda(**{"$select": "min(latitude) as lat_min, max(latitude) as lat_max, "
                           "min(longitude) as lon_min, max(longitude) as lon_max"}))

,lat_min,lat_max,lon_min,lon_max
0,0,51.212357,-114.304172,0


In [61]:
# Rows with zero coordinates
for label, where in [("latitude = 0", "latitude = 0"), ("longitude = 0", "longitude = 0")]:
    print(label, soda(**{"$select": "count(*) as n", "$where": where})["n"][0])

latitude = 0 921
longitude = 0 921


In [62]:
# Zero coordinates by permit type
display(soda(**{"$select": "permittype, count(*) as n",
                "$where": "latitude = 0 or longitude = 0",
                "$group": "permittype", "$order": "n DESC"}))

,permittype,n
0,Residential Improvement Project,380
1,Single Construction Permit,268
2,Commercial / Multi Family Project,224
3,Demolition,43
4,Environmental Restoration Permit,6


In [63]:
# Minimum and maximum of latitude and longitude, leaving out the rows with zeros
display(soda(**{"$select": "min(latitude) as lat_min, max(latitude) as lat_max, "
                           "min(longitude) as lon_min, max(longitude) as lon_max",
                "$where": "latitude != 0 and longitude != 0"}))

,lat_min,lat_max,lon_min,lon_max
0,50.848554,51.212357,-114.304172,-113.860656


**Result:**
- 921 rows (0.18%) have latitude 0 and longitude 0. The counts for latitude = 0 and longitude = 0 are both 921, so these are the same rows.
- By permit type: Residential Improvement Project 380, Single Construction Permit 268, Commercial / Multi Family Project 224, Demolition 43 and Environmental Restoration Permit 6. Within each type the share is between 0.17% and 0.24%, except Environmental Restoration Permit (0.8%, 6 rows).
- Without those rows, latitude runs from 50.848554 to 51.212357 and longitude from -114.304172 to -113.860656, inside the expected extent of Calgary. No point falls outside the city.

**Notes:**
- A coordinate of 0 is a placeholder and not a real location. These rows have values in `latitude` and `longitude`, but they cannot be placed on a map.
- These rows have the community name "N/A" (see 0.11.1), so they have no usable location in this table.

### 0.11.1 Community names of the rows with placeholder coordinates

`communityname` has no null values, so the applications with placeholder coordinates do carry a community name. This check lists the most frequent names among those rows, to see whether they are real communities or a generic label.

In [64]:
# Most frequent community names among the applications with placeholder coordinates
display(soda(**{"$select": "communityname, count(*) as n", "$where": "latitude = 0",
                "$group": "communityname", "$order": "n DESC", "$limit": 15}))

,communityname,n
0,NaN,921


In [65]:
# Check how the blank community names are stored: as null or as empty text, and whether they match latitude = 0
for label, where in [("communityname IS NULL", "communityname IS NULL"),
                     ("communityname = ''", "communityname = ''"),
                     ("blank communityname and latitude = 0", "communityname = '' AND latitude = 0")]:
    print(label, soda(**{"$select": "count(*) as n", "$where": where})["n"][0])

communityname IS NULL 0
communityname = '' 0
blank communityname and latitude = 0 0


In [66]:
# Read the raw text of the community name for these rows, without pandas converting "NA" style values to NaN
raw = requests.get(BASE, params={"$select": "communityname, count(*) as n", "$where": "latitude = 0",
                                 "$group": "communityname", "$limit": 5}, timeout=120).text
print(raw)

"communityname","n"
"N/A","921"



In [67]:
# Count all rows with the community name "N/A", and read the raw values of workclassmapped
print("communityname = 'N/A':", soda(**{"$select": "count(*) as n", "$where": "communityname = 'N/A'"})["n"][0])
print(requests.get(BASE, params={"$select": "workclassmapped, count(*) as n", "$group": "workclassmapped"}, timeout=120).text)

communityname = 'N/A': 921
"workclassmapped","n"
"Existing","252118"
"New","245941"
,"4755"



**Result:**
- The 921 rows with placeholder coordinates all have the community name "N/A". The value is stored as text, not as null or empty text, and pandas reads it as `NaN` by default.
- The label "N/A" appears in `communityname` only on those 921 rows.
- The 4,755 missing values of `workclassmapped` are empty in the raw text, so they are true blanks and not a label.

**Notes:**
- These applications have neither coordinates nor a community name, so they cannot be placed geographically from the data in this table.
- The helper `soda()` reads the CSV with the pandas defaults, so a text value such as "N/A" appears as `NaN`. When the difference matters, the raw text is read directly, as in the last cell.

## 0.12 Location columns

Two of the location columns were not inspected in section 0.11. The first check shows how many applications have each number of linked locations(`locationcount`), grouped into ranges. The second compares missing values in `point` with missing values in `latitude`, to see whether the two columns carry the same information. It checks only which values are present, not that their values match.

In [68]:
# Count applications by number of linked locations, grouped into ranges
lc = soda(**{"$select": "locationcount, count(*) as n", "$group": "locationcount",
             "$order": "locationcount", "$limit": 5000})
ranges = pd.cut(lc["locationcount"], [0, 1, 2, 3, 5, 10, 100, float("inf")],
                labels=["1", "2", "3", "4 to 5", "6 to 10", "11 to 100", "more than 100"])
summary = lc.groupby(ranges, observed=True)["n"].sum().to_frame("applications")
summary["share_pct"] = (summary["applications"] / lc["n"].sum() * 100).round(1)
display(summary)

# Missing values and the largest number of linked locations
print("missing:", lc.loc[lc["locationcount"].isna(), "n"].sum(), "| maximum:", lc["locationcount"].max())

,applications,share_pct
locationcount,,
1,200671,39.9
2,209151,41.6
3,70649,14.1
4 to 5,10897,2.2
6 to 10,6451,1.3
11 to 100,3646,0.7
more than 100,428,0.1


missing: 921 | maximum: 638.0


In [69]:
# Compare the presence of the point column with latitude, including the placeholder zeros
checks = [("point missing", "point IS NULL"),
          ("latitude missing", "latitude IS NULL"),
          ("point missing but latitude present", "point IS NULL AND latitude IS NOT NULL"),
          ("point present but latitude missing", "point IS NOT NULL AND latitude IS NULL"),
          ("point present with latitude = 0", "point IS NOT NULL AND latitude = 0")]
for label, where in checks:
    print(label, soda(**{"$select": "count(*) as n", "$where": where})["n"][0])

point missing 0
latitude missing 0
point missing but latitude present 0
point present but latitude missing 0
point present with latitude = 0 921


**Result:**
- `locationcount` has 921 missing values. 39.9% of applications link to one location, 41.6% to two and 18.3% to three or more (the rounded shares in the table add to 18.4%), up to 638 locations.
- `point` has a value in every row. No row is missing `point` or `latitude`, and 921 rows have a `point` value together with latitude 0.

**Notes:**
- Most applications are linked to more than one location. The portal does not explain what a location is in this context, so the high counts (hundreds of locations) are kept as provided and not interpreted.
- Each application has one coordinate pair, so maps and spatial analysis place each application at a single point, even when several locations are linked to it.
- `point` is present for the placeholder rows, so it cannot be used to recover their location. The number of missing `locationcount` values (921) equals the number of rows with placeholder coordinates, which suggests they are the same rows. This is not verified.

## 0.13 Classification columns: raw, grouped and mapped

The table describes each application with three classification dimensions: permit type, permit class and work class. Each dimension is stored in up to three versions: the raw value, a grouped version (`...group`) and a mapped version (`...mapped`). Before choosing which version to use in the analysis, the first step is to count how many distinct values each column has. A column with few values supports clear summaries by category. A column with many values may need grouping.

In [70]:
# Count the distinct values in each classification column (raw, grouped and mapped versions)
class_cols = ["permittype", "permittypemapped",
              "permitclass", "permitclassgroup", "permitclassmapped",
              "workclass", "workclassgroup", "workclassmapped"]
select_expr = ", ".join(f"count(distinct {c}) as {c}" for c in class_cols)
display(soda(**{"$select": select_expr}).T.rename(columns={0: "distinct values"}))

,distinct values
permittype,5
permittypemapped,2
permitclass,325
permitclassgroup,16
permitclassmapped,3
workclass,10
workclassgroup,4
workclassmapped,2


**Result:**

| Dimension | Raw | Grouped | Mapped |
|---|---|---|---|
| Permit type | 5 | not available | 2 |
| Permit class | 325 | 16 | 3 |
| Work class | 10 | 4 | 2 |

**Notes:**
- Raw `permitclass` has 325 distinct values, which is too many for summaries by category. The grouped version (16 values) and the mapped version (3 values) are examined in the next checks.
- The other raw columns (`permittype` with 5 values and `workclass` with 10) are small enough to use directly.
- The portal does not document the meaning of the mapped versions or their relationship with the raw and grouped values. This is reviewed in the next checks.

In [71]:
# Show the values of the three mapped columns with the number of applications in each
for col in ["permittypemapped", "permitclassmapped", "workclassmapped"]:
    print(f"\n{col}")
    display(soda(**{"$select": f"{col}, count(*) as n", "$group": col, "$order": "n DESC"}))



permittypemapped


,permittypemapped,n
0,Building,485134
1,Demolition,17680



permitclassmapped


,permitclassmapped,n
0,Residential,413578
1,Non-Residential,84904
2,Unspecified,4332



workclassmapped


,workclassmapped,n
0,Existing,252118
1,New,245941
2,NaN,4755


**Result:**
- `permittypemapped`: Building (485,134; 96.5%) and Demolition (17,680; 3.5%).
- `permitclassmapped`: Residential (413,578; 82.3%), Non-Residential (84,904; 16.9%) and Unspecified (4,332; 0.9%).
- `workclassmapped`: Existing (252,118; 50.1%), New (245,941; 48.9%) and 4,755 missing values (0.9%).

**Notes:**
- The mapped versions reduce each dimension to two or three values, which suits high level summaries. The grouped versions keep more detail (16 values for permit class, 4 for work class).
- `Unspecified` in `permitclassmapped` is an explicit category and is not the same as a missing value. In `workclassmapped`, missing values are true blanks.

In [72]:
# List all values of permitclassgroup with their number of applications and share of the table
groups = soda(**{"$select": "permitclassgroup, count(*) as n", "$group": "permitclassgroup", "$order": "n DESC"})
groups["pct"] = (groups["n"] / groups["n"].sum() * 100).round(2)
display(groups)


,permitclassgroup,n,pct
0,Single Family,273258,54.35
1,Garage,64418,12.81
2,Commercial,62472,12.42
3,Two Family,28406,5.65
4,Secondary Suites,18523,3.68
5,Unspecified,13686,2.72
6,Townhouse,10889,2.17
7,Institutional,9695,1.93
8,Apartment,7002,1.39
9,Industrial,5596,1.11


**Result:**
- The 16 groups add up to the full table. No row has a missing `permitclassgroup`.
- The five largest groups account for 88.9% of the applications: Single Family (54.3%), Garage (12.8%), Commercial (12.4%), Two Family (5.6%) and Secondary Suites (3.7%).
- Six groups each hold less than 1% of the table, and together 1.8%. The smallest are Swimming Pool (626 rows) and Retaining Wall (98 rows).
- `Unspecified` has 13,686 rows (2.7%) at group level, while `permitclassmapped` has only 4,332 `Unspecified` rows. Part of these applications is therefore assigned a class in the mapped version.
- `Special Function Ten` has 3,521 rows (0.7%).

**Notes:**
- The groups mix two kinds of categories: building use (Single Family, Commercial, Institutional) and type of structure (Garage, Swimming Pool, Retaining Wall). They are not a single taxonomy, which has to be kept in mind when reading comparisons by class.
- Groups with few applications give unstable percentiles, which matters for the cost analysis (Question 2).
- The portal does not define `Special Function Ten`. The name resembles the City's "Special Function" category, which in its development permit documents refers to temporary event setups such as tents and stages. This reading is plausible but not confirmed.

In [73]:
# Cross-tabulate the grouped class with the mapped class to see how each group is assigned
display(soda(**{"$select": "permitclassgroup, permitclassmapped, count(*) as n",
                "$group": "permitclassgroup, permitclassmapped",
                "$order": "permitclassgroup, n DESC"}))

,permitclassgroup,permitclassmapped,n
0,Additional Dwelling,Residential,1152
1,Apartment,Residential,7002
2,Commercial,Non-Residential,62472
3,Garage,Residential,64418
4,General,Non-Residential,2492
5,Government,Non-Residential,980
6,Industrial,Non-Residential,5596
7,Institutional,Non-Residential,9695
8,Retaining Wall,Non-Residential,98
9,Secondary Suites,Residential,18523


**Result:**
- Every group maps to a single mapped class, except `Unspecified`.
- Residential: Additional Dwelling, Apartment, Garage, Secondary Suites, Single Family, Swimming Pool, Townhouse and Two Family.
- Non-Residential: Commercial, General, Government, Industrial, Institutional, Retaining Wall and Special Function Ten.
- The 13,686 applications grouped as `Unspecified` are split in the mapped version: 9,304 Residential, 4,332 Unspecified and 50 Non-Residential.

**Notes:**
- `permitclassmapped` is a coarser version of `permitclassgroup`. Results by group can therefore be added up to the mapped level.
- Most `Unspecified` applications (9,354 of 13,686) receive a residential or non-residential class in the mapped version, so the mapping must use information other than the group. The rule is not documented in the portal.
- After mapping, 4,332 applications (0.9% of the table) remain without a residential or non-residential class.

## 0.14 Status roles

The City does not publish definitions for the status values. To interpret them without relying on assumptions, each status is described by what the dates show: how many applications carry it, and what share of them has an issue date and a completion date. A status where most applications have an issue date but no completion date behaves like an issued, work in progress status. A status with no issue date behaves like a pre issuance stage or a closed without issue outcome.

In [74]:
# Count applications per status and how many of them have an issue date and a completion date
roles = soda(**{"$select": "statuscurrent, count(*) as n, count(issueddate) as n_issued, count(completeddate) as n_completed",
                "$group": "statuscurrent", "$order": "n DESC"})

# Convert the date counts into percentages of each status
roles["pct_issued"] = (roles["n_issued"] / roles["n"] * 100).round(1)
roles["pct_completed"] = (roles["n_completed"] / roles["n"] * 100).round(1)
display(roles)

,statuscurrent,n,n_issued,n_completed,pct_issued,pct_completed
0,Completed,455111,454917,455086,100.0,100.0
1,Cancelled,14886,2591,14871,17.4,99.9
2,Issued Permit,11036,11036,19,100.0,0.2
3,Expired,7611,7526,7610,98.9,100.0
4,File Closed,4463,4450,4462,99.7,100.0
5,Pre Backfill Phase,2404,2404,0,100.0,0.0
6,Hold,1903,112,10,5.9,0.5
7,Pending Plans Review Assignment,1035,0,0,0.0,0.0
8,In Review,914,68,21,7.4,2.3
9,Refused,874,1,874,0.1,100.0


**Result:** the statuses show four patterns in their dates.

| Pattern | Statuses | Rows | Share |
|---|---|---|---|
| Issue date and completion date | Completed, Expired, File Closed, Suspended-Expired | 467,190 | 92.9% |
| Issue date, no completion date | Issued Permit, Pre Backfill Phase, Pre Board Phase, Final Phase, Re-instated, Work Restarted | 14,277 | 2.8% |
| Completion date, mostly no issue date | Cancelled, Refused | 15,760 | 3.1% |
| Neither date, mostly | Hold and its variants, Pending Plans Review Assignment, In Review, New, Pending Plans Review, Application Received, Pending Prescreen Review, Pending Completeness Review, Pending - Building By-Law Review | 5,587 | 1.1% |

**Notes:**
- The City does not publish definitions for the statuses. These patterns describe what the dates show, and they are not an official classification.
- Statuses with an issue date but no completion date (the second pattern) appear to sit between issue and completion. This is an inference from the data.
- 2,591 of the 14,886 Cancelled applications (17.4%) have an issue date, so they were issued before being cancelled.
- A few rows are inconsistent with their pattern, for example 68 In Review and 112 Hold applications with an issue date, 19 Issued Permit applications with a completion date, and 194 Completed applications without an issue date.
- Applications in the last pattern have not been issued yet, which matters for Question 1 (right-censoring). Those in the third pattern ended without a permit, which acts as a competing outcome.

## Data dictionary

The tables below document the 30 columns of the dataset in blocks by theme. Types are those reported by the portal. Where the portal gives no description, the meaning is stated as read from the column name and the data, and is marked as such. The findings column summarizes the audit results from the previous sections.

### Identifier, status and dates

| Column | Type | Role | Meaning | Audit findings |
|---|---|---|---|---|
| `permitnum` | text | Identifier | Number that identifies the application. | Unique: no value repeats and none is missing (see 0.2.1). |
| `statuscurrent` | text | Status | Current status of the application. | 24 values. The City does not publish definitions. The dates show four patterns (see 0.14), and the grouping into roles is decided in Phase 1. |
| `applieddate` | calendar date | Date | Date the application was submitted. Start of the time to issue. | No missing values. Records start in June 1999. 1999 and 2026 are partial years. |
| `issueddate` | calendar date | Date | Date the permit was issued. End of the time to issue. | 18,863 missing (3.8%), mostly applications that were not issued. 292 closed applications (Completed 194, Expired 85, File Closed 13) have no issue date. 105,304 applications (20.9%) were issued on the day they were submitted. |
| `completeddate` | calendar date | Date | Date the application was completed or closed. | 19,852 missing (3.9%). Also marks the closing date for terminal statuses such as Cancelled, Refused, Expired and File Closed. 31 applications have a completion date earlier than the issue date. |

### Classification

| Column | Type | Role | Meaning | Audit findings |
|---|---|---|---|---|
| `permittype` | text | Classification | Type of permit application. | 5 values: Residential Improvement Project (45.1%), Single Construction Permit (31.1%), Commercial / Multi Family Project (20.1%), Demolition (3.5%) and Environmental Restoration Permit (0.1%). Small enough to use directly. |
| `permittypemapped` | text | Classification | Simplified version of the permit type. | 2 values: Building (96.5%) and Demolition (3.5%). |
| `permitclass` | text | Classification | Class of the permit, read from the column name. | 325 distinct values, too many for summaries by category. |
| `permitclassgroup` | text | Classification | Grouped version of the permit class. | 16 values. The five largest cover 88.9% of the table. It mixes building use (Single Family, Commercial) with type of structure (Garage, Swimming Pool). `Special Function Ten` is not defined by the portal. |
| `permitclassmapped` | text | Classification | Simplified version of the permit class. | 3 values: Residential (82.3%), Non-Residential (16.9%) and Unspecified (0.9%). A coarser version of `permitclassgroup`. The rule that assigns a class to most `Unspecified` groups is not documented. |
| `workclass` | text | Classification | Kind of work carried out, read from the column name. | 10 values. Small enough to use directly. |
| `workclassgroup` | text | Classification | Grouped version of the work class. | 4 values: New (48.9%), Improvement (46.6%), Demolition (3.5%) and Unspecified (0.9%). The count of `Unspecified` (4,755) equals the number of missing values in `workclassmapped`. |
| `workclassmapped` | text | Classification | Simplified version of the work class. | 2 values: Existing (50.1%) and New (48.9%). 4,755 applications (0.9%) are missing. |

### Cost and size

| Column | Type | Role | Meaning | Audit findings |
|---|---|---|---|---|
| `housingunits` | number | Measure | Number of housing units in the project, read from the column name. | No missing values. 312,857 zeros (62.2%) and 180,073 ones (35.8%); 2 negative values (minimum -1); maximum 715. Zero probably means no housing units (not confirmed). |
| `estprojectcost` | number | Measure | Estimated cost of the project. The portal does not state the currency. | 39,023 missing values (7.8%) and no zeros, which may mean that projects without an estimate are stored as missing (not confirmed). 8 negative values, all Completed applications from 2004 to 2012, are flagged. Maximum 868,255,213; the ten highest values (315,000,000 and above) all belong to Commercial / Multi Family projects. |
| `totalsqft` | number | Measure | Total area of the project, read from the column name. | Structurally missing: it is recorded only for one permit type (Single Construction Permit), so it is analyzed within that type. Within that type, 652 values are zero and 2 are negative (minimum -15,313; maximum 41,979.25). |

### Location: address and community

| Column | Type | Role | Meaning | Audit findings |
|---|---|---|---|---|
| `originaladdress` | text | Location | Address of the application, read from the column name. | Not inspected. Location analysis relies on community and coordinates. |
| `communitycode` | text | Location | Code of the community. | Not inspected. |
| `communityname` | text | Location | Name of the community. Main geographic unit for Question 4. | No null or empty values. 921 rows, the same ones with placeholder coordinates, carry the label "N/A" instead of a community name. |
| `locationaddresses` | text | Location | Addresses of the locations linked to the application, read from the column name. | Not inspected. Related to `locationcount`. |

### Location: coordinates and geometry

| Column | Type | Role | Meaning | Audit findings |
|---|---|---|---|---|
| `latitude` | number | Location | Latitude of the application. | 921 applications (0.18%) have latitude and longitude both equal to 0, a placeholder for missing coordinates. The remaining values range from 50.848554 to 51.212357, inside the extent of Calgary. |
| `longitude` | number | Location | Longitude of the application. | Same 921 placeholder rows. The remaining values range from -114.304172 to -113.860656. |
| `point` | point | Location | Point geometry of the application. | Present in every row, including the rows with placeholder coordinates, so it is not an independent source of location for them. |
| `locationcount` | number | Location | Number of locations linked to the application, read from the column name. | Missing in 921 applications, the same count as the placeholder coordinates. 39.9% of applications have one linked location, 41.6% two and 18.3% three or more, up to 638. |
| `locationtypes` | text | Location | Types of the linked locations, read from the column name. | Not inspected. |
| `locationswkt` | text | Geometry | Geometry of the linked locations as text (WKT). | Not inspected. Long text, probably not needed for the analysis. |
| `locationsgeojson` | text | Geometry | Geometry of the linked locations as text (GeoJSON). | Not inspected. Long text, probably not needed for the analysis. |

### Free text

| Column | Type | Role | Meaning | Audit findings |
|---|---|---|---|---|
| `description` | text | Text | Free text description of the project. | Not inspected. Not used in the analysis. |
| `applicantname` | text | Text | Name of the applicant, read from the column name. | Not inspected. Not used in the analysis. Names are not shown in the report. |
| `contractorname` | text | Text | Name of the contractor, read from the column name. | Not inspected. Not used in the analysis. Names are not shown in the report. |

## Issues log

The audit found the data quality issues and structural features listed below. Each entry states its type, how many rows it affects and which of the five questions it touches (Q1 processing time, Q2 project cost, Q3 volume over time, Q4 geography, Q5 missingness). Nothing has been changed in the data at this stage. The log is the input to Phase 1, where the handling of each entry is decided.

| # | Finding | Type | Rows | Affects |
|---|---|---|---|---|
| 1 | Negative estimated costs | Anomaly | 8 | Q2 |
| 2 | Extremely high estimated costs (maximum 868,255,213) | Extreme value | Ten highest: 315,000,000 to 868,255,213 | Q2 |
| 3 | Missing estimated cost | Missing | 39,023 (7.8%) | Q2, Q5 |
| 4 | `totalsqft` recorded only for one permit type | Structure | 346,307 missing (68.9%) | Q2 |
| 5 | Coordinates equal to 0 | Placeholder | 921 (0.18%) | Q4 |
| 6 | Several linked locations, one coordinate pair | Structure | 301,222 (59.9%) with 2 or more | Q4 |
| 7 | Closed applications with no issue date | Anomaly | 292 | Q1 |
| 8 | Time to issue exists only for issued applications | Structure | 483,951 issued (96.2%) | Q1 |
| 9 | Cancelled applications that had been issued | Structure | 2,591 | Q1 |
| 10 | Statuses inconsistent with their dates | Anomaly | For example 68 In Review and 112 Hold with an issue date, and 25 Completed without a completion date | Q1 |
| 11 | Status definitions not published | Limitation | 24 values | Q1, Q3 |
| 12 | Raw permit class too detailed, and unspecified classes | Structure | 325 values; `Unspecified` 13,686 (2.7%) | Q2, Q4 |
| 13 | `workclassmapped` missing | Missing | 4,755 (0.9%) | Q5 |
| 14 | `Special Function Ten` is not defined | Limitation | 3,521 (0.7%) | Q2 |
| 15 | Partial years (1999 and 2026) | Structure | Whole years | Q3 |
| 16 | Currency of costs not stated | Limitation | All costs | Q2 |
| 17 | Table refreshed during the audit | Limitation | 129 rows added (502,685 to 502,814) | All |
| 18 | Dates out of order (issued before applied: 1; completed before issued: 31; completed before applied: 1) | Anomaly | 33 at most | Q1 |
| 19 | Issued on the same day as applied (time to issue of zero days) | Structure | 105,304 (20.9%) | Q1 |
| 20 | `totalsqft` with zero or negative values within Single Construction Permit | Anomaly | 652 zeros (0.4%) and 2 negative | Q2, Q5 |
| 21 | `housingunits` is zero for most rows, and 2 values are negative | Structure | 312,857 zeros (62.2%); 2 negative | Q2 |
| 22 | Community name "N/A" (same rows as the placeholder coordinates) | Placeholder | 921 (0.18%) | Q4 |

## Closing summary



### What the audit established
- The table holds one row per building permit application, identified by a unique `permitnum`: about 503,000 rows and 30 columns, with records from 1999-06-22 onward. The table keeps growing during the day, so Phase 1 saves a dated snapshot.
- The key columns are largely complete. The main gaps are the estimated cost (7.8% missing), `totalsqft` (recorded only for Single Construction Permit) and the issue and completion dates, whose missing values depend on the status of the application.
- Most findings affect Question 1 (time to issue) and Question 2 (project cost). The issues log lists 22 entries, each with its type, size and the questions it touches.

### Definitions used in the analysis
- **Time to issue:** the number of days between `applieddate` and `issueddate`. It exists only for applications with an issue date. About one in five applications was issued on the day it was submitted, so a time of zero days is common.
- **Right-censored application:** an application with no issue date that is still open (in review, pending or on hold). It is observed only up to the date of the snapshot, and its final outcome is unknown.
- **Competing outcome:** an application that ended without a permit (Cancelled or Refused). It can no longer be issued, so it is not the same as a censored application.
- **Time to close:** the number of days between `applieddate` and `completeddate`, for any status that has a completion date.

### Carried to Phase 1

The handling of each entry in the issues log is decided in Phase 1. The points to settle are:
- **Anomalies:** negative estimated costs, zero or negative `totalsqft`, negative `housingunits`, coordinates equal to 0 with the community name "N/A", closed applications without an issue date, and dates out of order.
- **Status grouping:** a derived variable `status_group` with six groups: Completed; Ended without issue (Cancelled, Refused); Issued, in progress; Closed after issue (Expired, File Closed, Suspended-Expired); Open before issue; and On hold.
- **Classification level:** `permittype` for Questions 1 and 5, `permitclassgroup` for Question 2, `permittypemapped` for Question 3 and `permitclassmapped` for Question 4, which is also a second view for Questions 1 and 3.
- **Snapshot:** download the chosen columns and save them as a dated file in the project repository on GitHub.

### Limitations
- The City does not publish definitions for the statuses or for the `Special Function Ten` group. The interpretations in this notebook are inferences from the data.
- The currency of the cost values is not stated.
- Phase 0 queries the live table. The figures in this notebook describe the version available when it was run, on the date shown in "Data as of".